# Task 1, Steps 8-9: 2:4 Diagnostic, Resource-Quality Trade-offs, Qualitative Cases

* **2:4 vs unstructured at 50% Wanda:** keep the two highest Wanda scores in every group of four contiguous
  input-channel weights. Identical eligible sparsity (exactly 50%). Quality and representation study only;
  nothing here is executed with sparse tensor cores.
* **Trade-offs:** quality vs eligible sparsity (pruning) and vs packed bytes (quantization), plotted separately.
* **Calibration vs held-out:** do calibration / development improvements carry over to WikiText-2, C4 and HellaSwag?
* **Theoretical vs measured savings:** storage formats vs dense-FP16 latency and peak memory actually measured.
* **Qualitative:** a fixed rule selects HellaSwag items; the same candidate scores are shown for every condition.

In [ ]:
# --- Setup: locate (or clone) the PA4 repo, pin library versions, prepare data -------------
import os, sys, subprocess, importlib.metadata as md
REPO_URL = "https://github.com/moizejaz-31/ATML-PA4.git"   # change if your repo lives elsewhere

def _find_root(d):
    d = os.path.abspath(d)
    while True:
        if os.path.isdir(os.path.join(d, "shared")) and os.path.isdir(os.path.join(d, "task1_compression")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

ROOT = _find_root(os.getcwd())
if ROOT is None:                                   # fresh Kaggle / Colab runtime
    subprocess.run(["git", "clone", "-q", REPO_URL, "ATML-PA4"], check=True)
    ROOT = os.path.abspath("ATML-PA4")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

if _ver("transformers") != "4.57.1" or _ver("huggingface_hub") != "0.36.2":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "huggingface_hub==0.36.2",
                    "datasets>=3,<5"], check=True)
    print(">>> Installed pinned versions. Restart the kernel, then run this cell again.")

from shared.src.nb import init_notebook
ctx = init_notebook("task1", need_public=True, need_prefix_banks=False)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from shared.src.plotting import save_figure
from shared.src.io import load_json, save_json
from task1_compression.src.runner import setup_task1
from task1_compression.src import prune
from task1_compression.src.analysis import summary_table, compare_table, load_condition, per_example_hs, available

T = setup_task1(ctx)
T.runner.run("p2_wanda_24", apply_fn=lambda m: prune.prune_sequential(m, T.calib, "wanda", None, nm=(2, 4), gram=True),
             meta=dict(method="wanda 2:4 (keep 2 of every 4 contiguous input channels)", target_sparsity=0.5))

In [ ]:
compare_table(ctx.results, [("p2_wanda_24", "p2_wanda_s50"), ("p2_wanda_24", "fp16")])

In [ ]:
a = load_condition(ctx.results, "p2_wanda_24")["compression"]["stats"]
b = load_condition(ctx.results, "p2_wanda_s50")["compression"]["stats"]
print("zeros 2:4", a["zeros"], "unstructured", b["zeros"])
err = pd.DataFrame({"2:4": [x["block_out_rel_err"] for x in a["per_block"]], "unstructured": [x["block_out_rel_err"] for x in b["per_block"]]})
err.plot(logy=True, figsize=(9, 3.5), xlabel="block", ylabel="block output rel. error (calibration)")
save_figure(ctx.figures, "wanda_24_vs_unstructured_block_error")
# 2:4 storage: 2 values of 16 bits + 2-bit index per kept value (4 bits metadata per group of 4) vs the bitmask format
n_elig = load_condition(ctx.results, "fp16")["nonzeros"]["n_eligible"]
print(f"2:4 format (FP16 values + 2-bit indices): {(n_elig/2*2 + n_elig/4*4/8)/2**20:.1f} MiB eligible;"
      f" bitmask format: {(n_elig/8 + n_elig/2*2)/2**20:.1f} MiB")

## Summary of all conditions with paired changes from FP16

In [ ]:
tab = summary_table(ctx.results)
tab.to_csv(ctx.results / "summary_all_conditions.csv")
tab

In [ ]:
pairs = [(n, "fp16") for n in available(ctx.results) if n != "fp16"]
ct = compare_table(ctx.results, pairs)
ct.to_csv(ctx.results / "paired_vs_fp16.csv", index=False)
ct

## Resource-quality trade-offs (pruning and quantization separated)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
groups = {"P1 magnitude": ["p1_mag_s25", "p1_mag_s50", "p1_mag_s75"], "P2 Wanda": ["p2_wanda_s25", "p2_wanda_s50", "p2_wanda_s75"],
          "P3 nonuniform": ["p3_s50", "p3_s75"], "Wanda 2:4": ["p2_wanda_24"]}
for lab, names in groups.items():
    names = [n for n in names if n in tab.index]
    axes[0, 0].plot(tab.loc[names, "sparsity"], tab.loc[names, "wikitext2_ppl"], "o-", label=lab)
    axes[0, 1].plot(tab.loc[names, "sparsity"], tab.loc[names, "hs_acc_norm"], "o-", label=lab)
qgroups = {"Q1 uniform": ["q1_int8", "q1_int4", "q1_int3"], "Q2 AWQ": ["q2_awq_int8", "q2_awq_int4", "q2_awq_int3"],
           "Q3 mixed": ["q3_budget_int4", "q3_budget_int3"]}
for lab, names in qgroups.items():
    names = [n for n in names if n in tab.index]
    axes[1, 0].plot(tab.loc[names, "packed_MB"], tab.loc[names, "wikitext2_ppl"], "o-", label=lab)
    axes[1, 1].plot(tab.loc[names, "packed_MB"], tab.loc[names, "hs_acc_norm"], "o-", label=lab)
for ax in axes[:, 0]:
    ax.axhline(tab.loc["fp16", "wikitext2_ppl"], color="k", ls="--", lw=1); ax.set_yscale("log"); ax.set_ylabel("WikiText-2 PPL")
for ax in axes[:, 1]:
    ax.axhline(tab.loc["fp16", "hs_acc_norm"], color="k", ls="--", lw=1); ax.set_ylabel("HellaSwag acc_norm")
for ax in axes[0]:
    ax.set_xlabel("eligible-weight sparsity"); ax.legend()
for ax in axes[1]:
    ax.set_xlabel("packed model bytes incl. metadata + FP16 non-eligible (MiB)"); ax.legend()
save_figure(ctx.figures, "tradeoffs_quality_vs_resources")

## Calibration error vs held-out quality, and dev-selection transfer

In [ ]:
rows = []
for n in available(ctx.results):
    r = load_condition(ctx.results, n)
    pb = r["compression"]["stats"].get("per_block") if isinstance(r["compression"]["stats"], dict) else None
    if pb:
        rows.append(dict(condition=n, mean_block_err=np.mean([b["block_out_rel_err"] for b in pb]),
                         final_block_err=pb[-1]["block_out_rel_err"], wiki_nll=r["wikitext2"]["mean_nll"], c4_nll=r["c4"]["mean_nll"],
                         hs_acc_norm=r["hellaswag"]["acc_norm"]))
cal = pd.DataFrame(rows).set_index("condition")
ax = cal.plot.scatter("final_block_err", "wiki_nll", logx=True, figsize=(6, 4))
for n, rr in cal.iterrows():
    ax.annotate(n, (rr.final_block_err, rr.wiki_nll), fontsize=7)
save_figure(ctx.figures, "calibration_error_vs_heldout_nll")
cal.corr(method="spearman")

In [ ]:
p3c, q3c = load_json(ctx.results / "p3_candidates.json"), load_json(ctx.results / "q3_candidates.json")
p3s, q3s = load_json(ctx.results / "p3_selection.json")["selected"], load_json(ctx.results / "q3_selection.json")["selected"]
transfer = []
for S, sel in p3s.items():
    S = float(S)
    d_dev = p3c[sel]["dev_nll"] - p3c[f"S{S}_uniform"]["dev_nll"]
    h = load_condition(ctx.results, f"p3_s{int(S*100)}")["wikitext2"]["mean_nll"] - load_condition(ctx.results, f"p2_wanda_s{int(S*100)}")["wikitext2"]["mean_nll"]
    c = load_condition(ctx.results, f"p3_s{int(S*100)}")["c4"]["mean_nll"] - load_condition(ctx.results, f"p2_wanda_s{int(S*100)}")["c4"]["mean_nll"]
    transfer.append(dict(comparison=f"P3-P2 @ {S}", dev_dNLL=d_dev, wiki_dNLL=h, c4_dNLL=c))
for b, sel in q3s.items():
    d_dev = q3c[sel]["dev_nll"] - q3c[f"int{b}_uniform_q2"]["dev_nll"]
    h = load_condition(ctx.results, f"q3_budget_int{b}")["wikitext2"]["mean_nll"] - load_condition(ctx.results, f"q2_awq_int{b}")["wikitext2"]["mean_nll"]
    c = load_condition(ctx.results, f"q3_budget_int{b}")["c4"]["mean_nll"] - load_condition(ctx.results, f"q2_awq_int{b}")["c4"]["mean_nll"]
    transfer.append(dict(comparison=f"Q3-Q2 @ INT{b} budget", dev_dNLL=d_dev, wiki_dNLL=h, c4_dNLL=c))
pd.DataFrame(transfer)

## Theoretical vs measured savings
Storage columns are logical/packed bytes; latency and peak memory are measured with dense FP16 kernels on the
fake-compressed weights, so they should not change with sparsity or nominal bits.

In [ ]:
res_cols = ["sparsity", "dense_fp16_MB", "sparse16_MB", "packed_MB", "prefill512_ms", "prefill2048_ms", "decode64_ms", "peak_alloc_MB", "compression_s"]
rt = tab[res_cols].copy()
rt["theoretical_storage_ratio"] = (rt["packed_MB"].fillna(rt["sparse16_MB"]) / rt["dense_fp16_MB"]).round(3)
rt["measured_prefill2048_ratio"] = (rt["prefill2048_ms"] / rt.loc["fp16", "prefill2048_ms"]).round(3)
rt.to_csv(ctx.results / "resources_theoretical_vs_measured.csv")
rt

## Qualitative HellaSwag cases (fixed selection rule)
Rule: among the first 512 IDs (scored by every condition), take the first 6 orders where at least one of the
50%-sparsity or INT3 conditions changes the FP16 acc_norm prediction, plus the first 2 orders where all agree.

In [ ]:
cmp = [n for n in ["p1_mag_s50", "p2_wanda_s50", "p3_s50", "p2_wanda_24", "q1_int3", "q2_awq_int3", "q3_budget_int3"] if n in tab.index]
hs = {n: per_example_hs(ctx.results, n) for n in ["fp16"] + cmp}
changed = [o for o in range(512) if any(hs[n][o]["pred_norm"] != hs["fp16"][o]["pred_norm"] for n in cmp)]
same = [o for o in range(512) if o not in changed]
pick = changed[:6] + same[:2]
docs = {d["order"]: d for d in T.ev["hellaswag"][:512]}
qual = []
for o in pick:
    d = docs[o]
    print(f"\n### order {o} | gold={d['gold']} | {d['query'][:160]}")
    for c, ch in enumerate(d["choices"]):
        print(f"   [{c}] {ch[:110]}")
    for n in ["fp16"] + cmp:
        e = hs[n][o]
        norm = (np.array(e["loglik"]) / np.array(e["n_chars"])).round(3).tolist()
        qual.append(dict(order=o, condition=n, pred_norm=e["pred_norm"], gold=e["gold"], loglik_per_char=norm))
        print(f"   {n:>15}: pred_norm={e['pred_norm']} {'OK ' if e['pred_norm'] == e['gold'] else 'XX '} ll/char={norm}")
save_json(ctx.results / "qualitative_hellaswag.json", dict(rule="first 6 changed + first 2 unchanged among orders 0-511", cases=qual))